In [1]:
# Install condacolab to enable the Bioconda package manager
!pip install -q condacolab
import condacolab
condacolab.install()



📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

✨🍰✨ Everything looks OK!


In [2]:
# Install the industry-standard sequence alignment and processing tools
!conda install -c bioconda samtools bwa fastqc bcftools -y


Retrieving notices: - \ | / - done
Channels:
 - bioconda
 - conda-forge
Platform: linux-64
Solving environment: / - \ | done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - bcftools
    - bwa
    - fastqc
    - samtools


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _python_abi3_support-1.0   |       hd8ed1ab_3           8 KB  conda-forge
    alsa-lib-1.2.16.1          |       h7cc23a3_1         581 KB  conda-forge
    annotated-types-0.8.0      |     pyhd8ed1ab_0          19 KB  conda-forge
    anyio-4.16.0               |     pyh5ded981_1         182 KB  conda-forge
    bcftools-1.24              |       h487d631_0         941 KB  bioconda
    bwa-0.7.19                 |       h577a1d6_1         639 KB  bioconda
    ca-certificates-2026.10.23 |       hbd8a1cb_0         127 KB  conda-forge
    cached-property-2.0.1      |    

In [3]:
# Create a data directory
!mkdir -p data

# Download a small bacterial reference genome (FASTA format)
!wget -O data/ref.fa https://githubusercontent.com || echo "Downloading placeholder data..."
# Let's generate a simulated small fasta and fastq setup for execution stability
with open("data/ref.fa", "w") as f:
    f.write(">chr_test\nATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCG\n")

with open("data/reads_R1.fq", "w") as f:
    f.write("@read1\nATCGATCGATCG\n+\nIIIIIIIIIIII\n")
with open("data/reads_R2.fq", "w") as f:
    f.write("@read1\nCGATCGATCGAT\n+\nIIIIIIIIIIII\n")
print("Environment dataset successfully generated.")


--2026-11-02 16:08:25--  https://githubusercontent.com/
Resolving githubusercontent.com (githubusercontent.com)... failed: No address associated with hostname.
wget: unable to resolve host address ‘githubusercontent.com’
Environment dataset successfully generated.


In [4]:
%%bash
# 1. Evaluate sequencing quality
fastqc data/reads_R1.fq data/reads_R2.fq

# 2. Build index tracking vectors for the reference genome
bwa index data/ref.fa

# 3. Map raw sequencing strings to the structural reference
bwa mem data/ref.fa data/reads_R1.fq data/reads_R2.fq > data/aligned.sam

# 4. Compress and sort mapping vectors using Samtools
samtools view -S -b data/aligned.sam | samtools sort -o data/sorted.bam
samtools index data/sorted.bam

# 5. Extract structural genetic variants into an annotated VCF matrix
bcftools mpileup -f data/ref.fa data/sorted.bam | bcftools call -mv -Ob -o data/variants.bcf
bcftools view data/variants.bcf > data/final_variants.vcf

echo "Pipeline executed completely! Final variants saved to data/final_variants.vcf"


[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Analysis complete for reads_R1.fq
Analysis complete for reads_R2.fq
Pipeline executed completely! Final variants saved to data/final_variants.vcf


Started analysis of reads_R1.fq
Using Phred32 encoding your lowest quality is 41 could this file be Phred64 encoded?
Using Phred32 encoding your lowest quality is 41 could this file be Phred64 encoded?
Using Phred32 encoding your lowest quality is 41 could this file be Phred64 encoded?
Using Phred32 encoding your lowest quality is 41 could this file be Phred64 encoded?
Started analysis of reads_R2.fq
Using Phred32 encoding your lowest quality is 41 could this file be Phred64 encoded?
Using Phred32 encoding your lowest quality is 41 could this file be Phred64 encoded?
Using Phred32 encoding your lowest quality is 41 could this file be Phred64 encoded?
Using Phred32 encoding your lowest quality is 41 could this file be Phred64 encoded?
[bwa_index] Pack FASTA... 0.00 sec
[bwa_index] Construct BWT for the packed sequence...
[bwa_index] 0.00 seconds elapse.
[bwa_index] Update BWT... 0.00 sec
[bwa_index] Pack forward-only FASTA... 0.00 sec
[bwa_index] Construct SA from BWT and Occ... 0.00 se